# ADP PM Notebook 5 — Multi-Agent Workflows, Short Demo

**Purpose:** Keep only the product architecture idea of multi-agent workflows.



## PM mental model

A multi-agent workflow is useful when one task naturally needs different specialist roles, for example planning, writing, reviewing, and risk checking.

For this ADP PM session, treat this as a short product architecture pattern, not a full build lab.


In [ ]:
# Uncomment if needed.
# %pip install -q strands-agents boto3 pandas


In [ ]:
import boto3
import pandas as pd
from strands import Agent
from strands.models import BedrockModel

AWS_REGION = boto3.session.Session().region_name or "us-east-1"
MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION, temperature=0.2)
print("Region:", AWS_REGION)
print("Model:", MODEL_ID)


## 1. Specialist agents

Each agent has a narrow role. This makes behavior easier to describe, test, and govern.


In [ ]:
session_planner_agent = Agent(
    model=bedrock_model,
    system_prompt="You design concise training session plans for product managers.",
)

risk_checker_agent = Agent(
    model=bedrock_model,
    system_prompt="You review AI product ideas for data, privacy, hallucination, and approval risks.",
)

email_writer_agent = Agent(
    model=bedrock_model,
    system_prompt="You write short professional stakeholder emails.",
)

print("Specialist agents created")


## 2. Sequential multi-agent pipeline

In a sequential workflow, each agent's output becomes the next agent's input.


In [ ]:
product_idea = "Create an AI assistant that helps product managers search HR policy documents and draft backlog items."

plan = session_planner_agent(f"Create a short enablement plan for this product idea: {product_idea}")
print("STEP 1 — Planner output")
print(plan)

risk_review = risk_checker_agent(f"Review this plan for product risks:\n\n{plan}")
print("\nSTEP 2 — Risk checker output")
print(risk_review)

email = email_writer_agent(f"Write a short stakeholder update based on this plan and risk review:\n\nPlan:\n{plan}\n\nRisk review:\n{risk_review}")
print("\nSTEP 3 — Email writer output")
print(email)


## 3. Orchestrator concept

An orchestrator decides which specialist agent to call. In production, this is where routing, logging, fallback, and human approval become important.


In [ ]:
comparison = pd.DataFrame([
    {
        "Pattern": "Single agent",
        "Best for": "Simple tasks with one role",
        "PM concern": "Clear instructions and safe tools",
    },
    {
        "Pattern": "Sequential multi-agent",
        "Best for": "Fixed process such as draft → review → revise",
        "PM concern": "Quality gates between steps",
    },
    {
        "Pattern": "Orchestrator + specialists",
        "Best for": "Dynamic routing across specialist capabilities",
        "PM concern": "Routing accuracy, traceability, and fallback",
    },
])
display(comparison)


## PM reflection checkpoint

- When is a single agent enough?
- Which product workflows need specialist roles?
- Where should human approval sit?
- What logs are needed to explain agent routing?
